<a href="https://colab.research.google.com/github/yo-danny/goodread-fiction-classifier/blob/main/Goodreads_RNN_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classify a book into fiction or non-fiction

This notebook uses RNN (Recurrent Neural Networks) to classify books in fiction/non-fiction based on the description on Goodreads.

In [1]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=67f8f340b6983b396ddde599db4d718a49454816db4f4cd0a020a615976b4fca
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [2]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import numpy as np
import time
import os
import string
import plotly.offline as cf
import plotly.graph_objs as go
from plotly.offline import iplot, init_notebook_mode
init_notebook_mode(connected=True)

from langdetect import detect
from requests import get
from collections import defaultdict

from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.layers import Embedding
from keras.layers import LSTM

## Extracting data from Goodreads

Using BeautifulSoup to access the < td > tag in each row of the table we can get its URL

In [ ]:
# Header for not getting blocked
hdr = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.11 (KHTML, like Gecko) Chrome/23.0.1271.64 Safari/537.11',
         'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
         'Referer': 'https://cssspritegenerator.com',
         'Accept-Charset': 'ISO-8859-1,utf-8;q=0.7,*;q=0.3',
         'Accept-Encoding': 'none',
         'Accept-Language': 'en-US,en;q=0.8'}

BASE_URL = "https://www.goodreads.com"
LIST_URL = "https://www.goodreads.com/list/show/1.Best_Books_Ever"

# 'books' will store URLs for the current batch of pages
books = {"URL":[]}

# Num of pages this list has
num_pages = 557
init_page = 351

# Read existing data into books_df to append new data to it
try:
    books_df=pd.read_csv(r'/content/drive/MyDrive/Goodreads/books.csv')
except FileNotFoundError:
    books_df=pd.DataFrame(columns=["URL"])

for i in range (init_page, num_pages):
    time.sleep(20) # time to page load
    print(f"Reading page {i}")
    list_page_url = f"{LIST_URL}?page={i}"
    response = get(list_page_url, headers=hdr)
    html_soup = BeautifulSoup(response.content, 'html.parser')
    book_table = html_soup.find('table', attrs={'class':'tableList js-dataTooltip'})
    book_containers = book_table.find_all('tr')
    books['URL'].extend([BASE_URL+r.find('a', attrs={'class':'bookTitle'})['href'] for r in book_containers])

    # Periodically save the collected data
    if i%50 == 0:
      new_batch_df = pd.DataFrame.from_dict(books)
      books_df = pd.concat([books_df, new_batch_df], ignore_index=True)
      books_df.to_csv(r'/content/drive/MyDrive/Goodreads/books.csv', index=False)
      books = {"URL":[]}

# After the loop, if there's any remaining data in 'books', save it
if books['URL']:
    final_batch_df = pd.DataFrame.from_dict(books)
    books_df = pd.concat([books_df, final_batch_df], ignore_index=True)
    books_df.to_csv(r'/content/drive/MyDrive/Goodreads/books.csv', index=False)
    books = {"URL":[]}


In [ ]:
books = pd.read_csv(r'/content/drive/MyDrive/Goodreads/books.csv')

if not os.path.exists(r'/content/drive/MyDrive/Goodreads/book_data.csv'):
  book_data=pd.DataFrame(columns= [
      'image_url',
      'book_title',
      'book_authors',
      'book_rating',
      'book_rating_count',
      'book_review_count',
      'book_desc',
      'book_format',
      'book_edition',
      'book_pages',
      'book_isbn',
      'genres'
  ])

book_rows = []
save_every = 10

for i in range(len(book_data), len(books)):
  time.sleep(20)
  try:
    book_URL = books.at[i, 'URL']
    book_page = get(book_URL, headers=hdr)
    book_soup = BeautifulSoup(book_page.content, 'html.parser')

    book = dict()

    # cover of the book
    image_url = book_soup.find('img', attrs={'id':'coverImage'})
    if image_url:
       book['image_url'] = image_url.attrs['src']
    else:
      book['image_url'] = ''

    # title of the book
    book_title = book_soup.find('h1', attrs={'id':'bookTitle'})
    if book_title:
      book['book_title'] = book_title.text.replace('\n','').strip()
    else:
      book['book_title'] = ''

    #Author(s) of the book
    book['book_authors']='|'.join([a.find('span', attrs={'itemprop':'name'}).text for a in book_soup.find_all('a', attrs={'class':'authorName'})])

    #Rating given by users on goodreads
    book_rating=book_soup.find('span', attrs={'itemprop':'ratingValue'})
    if book_rating:
        book['book_rating']=book_rating.text.replace('\n','').strip()
    else:
        book['book_rating']=''
    book['book_rating_count']=book_soup.find('meta', attrs={'itemprop':'ratingCount'})['content']

    #No. of reviews for the book
    book['book_review_count']=book_soup.find('meta', attrs={'itemprop':'reviewCount'})['content']

    #A short description of the book, usually found on the back or inside cover of the book. Also called a blurb
    book_desc=book_soup.find('div', attrs={'class':'readable stacked'})
    if book_desc:
        book['book_desc']=book_desc.find_all('span')[-1].text
    else:
        book['book_desc']=''

    #Format of the book, e.g, paperback, hardcover, Kindle edition, etc.
    book_format=book_soup.find('div', attrs={'id':'details'}).find('span', attrs={'itemprop':'bookFormat'})
    if book_format:
        book['book_format']=book_format.text
    else:
        book['book_format']=''

    #Edition of the book
    book_edition=book_soup.find('div', attrs={'id':'details'}).find('span', attrs={'itemprop':'bookEdition'})
    if book_edition:
        book['book_edition']=book_edition.text
    else:
        book['book_edition']=''

    #No. of pages in the book
    book_pages=book_soup.find('div', attrs={'id':'details'}).find('span', attrs={'itemprop':'numberOfPages'})
    if book_pages:
        book['book_pages']=book_pages.text
    else:
        book['book_pages']=''

    #ISBN code of the book
    book_isbn=book_soup.find('div', attrs={'id':'bookDataBox'}).find('span', attrs={'itemprop':'isbn'})
    if book_isbn:
        book['book_isbn']=book_isbn.text
    else:
        book['book_isbn']=''

    #List of genres that the book belongs to. User supplied data.
    genres_list=book_soup.find_all('a', attrs={'class':'actionLinkLite bookPageGenreLink'})
    book['genres']='|'.join([i.text for i in genres_list])
    book_rows.append(book)

    if i%save_every==0:
        book_data.append(pd.DataFrame.from_dict(book_rows)).to_csv(r'/content/drive/MyDrive/Goodreads/book_data.csv', index=False)
        book_rows=[]
  except:
    book_data.append(pd.DataFrame.from_dict(book_rows)).to_csv(r'/content/drive/MyDrive/Goodreads/book_data.csv', index=False)
    book_rows=[]

## EDA (Exploratory Data Analysis)



In [3]:
book_data_train = pd.read_csv(r'/content/drive/MyDrive/Goodreads/book_data_train.csv')
book_data_test = pd.read_csv(r'/content/drive/MyDrive/Goodreads/book_data_test.csv')
book_data_train.head()

,book_authors,book_desc,book_edition,book_format,book_isbn,book_pages,book_rating,book_rating_count,book_review_count,book_title,genres,image_url
0,Suzanne Collins,Winning will make you famous. Losing means cer...,NaN,Hardcover,9.78044E+12,374 pages,4.33,5519135,160706,The Hunger Games,Young Adult|Fiction|Science Fiction|Dystopia|F...,https://images.gr-assets.com/books/1447303603l...
1,J.K. Rowling|Mary GrandPré,There is a door at the end of a silent corrido...,US Edition,Paperback,9.78044E+12,870 pages,4.48,2041594,33264,Harry Potter and the Order of the Phoenix,Fantasy|Young Adult|Fiction,https://images.gr-assets.com/books/1255614970l...
2,Harper Lee,The unforgettable novel of a childhood in a sl...,50th Anniversary,Paperback,9.78006E+12,324 pages,4.27,3745197,79450,To Kill a Mockingbird,Classics|Fiction|Historical|Historical Fiction...,https://images.gr-assets.com/books/1361975680l...
3,Jane Austen|Anna Quindlen|Mrs. Oliphant|George...,«È cosa ormai risaputa che a uno scapolo in po...,"Modern Library Classics, USA / CAN",Paperback,9.78068E+12,279 pages,4.25,2453620,54322,Pride and Prejudice,Classics|Fiction|Romance,https://images.gr-assets.com/books/1320399351l...
4,Stephenie Meyer,About three things I was absolutely positive.F...,NaN,Paperback,9.78032E+12,498 pages,3.58,4281268,97991,Twilight,Young Adult|Fantasy|Romance|Paranormal|Vampire...,https://images.gr-assets.com/books/1361039443l...


The genres on Goodreads are supplied by the users, this can led to multiple genres which could be duplicates or trivial.

In [4]:
genre_counts = defaultdict(int)
for i in book_data_train.index:
  g = book_data_train.at[i, 'genres']
  if type(g)==str:
    for j in g.split('|'):
      genre_counts[j]+=1

print(len(genre_counts))

866


## Data Cleaning

Excluding rows that don't have any genre or description, or that don't have the tagged genres of "fiction" or "non fiction".

Removed the non-English books from the dataset.

Plus some descriptions had formatting issues.

In [5]:
def genre_binarizer(genres):
  """
  Analyses the genres string passed as argument and classify into fiction or nonfiction
  """
  genre_list = genres.lower().split("|")
  if "fiction" in genre_list:
    return "fiction"
  elif "nonfiction" in genre_list:
    return "non-fiction"
  else:
    return "na"

def remove_non_english(df):
  """
  Removes records that have invalid descripstions from the dataset
  Input: dataframe
  Outout: cleaned dataframe
  """
  invalid_desc_indxs = []
  for i in df.index:
    try:
      a=detect(df.at[i,'book_desc'])
      if a!='en':
        invalid_desc_indxs.append(i)
    except:
      invalid_desc_indxs.append(i)
  df=df.drop(index=invalid_desc_indxs)
  return df

def add_space_case(desc):
  """
  Analyses the book description passed and inserts a space after a lowercase letter followed by a uppercase letter.
  """
  upd_desc = ''

  for i in range(len(desc)-1):
    upd_desc+=desc[i]
    if desc[i] in string.ascii_lowercase and desc[i+1] in string.ascii_uppercase:
      upd_desc+=' '
  upd_desc+=desc[-1]
  return upd_desc

def remove_punctuation(desc):
  """
  Modifies the book description passed to
  - insert spaces in place of punctuations
  - join apostrophe words to their parent words
  - insert spaces where lowercase is followed by uppercase
  """
  desc = add_space_case(desc)
  apostrophe_words = ["m","re","ve","ll","t","s","d"]

  desc = desc.lower()
  valid_chars = string.ascii_letters + string.digits + ' '
  desc="".join([c if c in valid_chars else "" for c in desc])

  for a in apostrophe_words:
    desc = desc.replace(""+a+"",a+"")

  return desc

def df_cleaner(df):
  """
  Performs the functions above on the dataset
  """
  df = df[df.genres.notnull()]
  df = df[df.book_desc.notnull()]
  df = df[df.book_desc.str.strip().apply(len)>0]
  df['binary_genre'] = df['genres'].apply(genre_binarizer)
  df = df[df.binary_genre!='na']
  df = remove_non_english(df)
  df["clean_desc"] = df.book_desc.apply(remove_punctuation)

  df.reset_index(drop=True, inplace=True)
  return df

book_data_train = df_cleaner(book_data_train)
book_data_test = df_cleaner(book_data_test)
book_data_train.head()

,book_authors,book_desc,book_edition,book_format,book_isbn,book_pages,book_rating,book_rating_count,book_review_count,book_title,genres,image_url,binary_genre,clean_desc
0,Suzanne Collins,Winning will make you famous. Losing means cer...,NaN,Hardcover,9.78044E+12,374 pages,4.33,5519135,160706,The Hunger Games,Young Adult|Fiction|Science Fiction|Dystopia|F...,https://images.gr-assets.com/books/1447303603l...,fiction,winning will make you famous losing means cert...
1,J.K. Rowling|Mary GrandPré,There is a door at the end of a silent corrido...,US Edition,Paperback,9.78044E+12,870 pages,4.48,2041594,33264,Harry Potter and the Order of the Phoenix,Fantasy|Young Adult|Fiction,https://images.gr-assets.com/books/1255614970l...,fiction,there is a door at the end of a silent corrido...
2,Harper Lee,The unforgettable novel of a childhood in a sl...,50th Anniversary,Paperback,9.78006E+12,324 pages,4.27,3745197,79450,To Kill a Mockingbird,Classics|Fiction|Historical|Historical Fiction...,https://images.gr-assets.com/books/1361975680l...,fiction,the unforgettable novel of a childhood in a sl...
3,Stephenie Meyer,About three things I was absolutely positive.F...,NaN,Paperback,9.78032E+12,498 pages,3.58,4281268,97991,Twilight,Young Adult|Fantasy|Romance|Paranormal|Vampire...,https://images.gr-assets.com/books/1361039443l...,fiction,about three things i was absolutely positivefi...
4,Markus Zusak,Trying to make sense of the horrors of World W...,First American Edition (US / CAN),Hardcover,9.78038E+12,552 pages,4.36,1485632,100821,The Book Thief,Historical|Historical Fiction|Fiction|Young Adult,https://images.gr-assets.com/books/1522157426l...,fiction,trying to make sense of the horrors of world w...


## Data Preprocessing

To fed the model we have to make sure the data is all of the same shape and lenght, to do so, we'll have to clip some of the descriptions to fit our model.

In [6]:
len_df = pd.DataFrame()
desc_lenghts = [len(i.split()) for i in book_data_train.clean_desc]
len_df["desc_lenghts"] = desc_lenghts

len_df_bins = len_df.desc_lenghts.value_counts(bins=100, normalize=True).reset_index().sort_values(by=["index"])
len_df_bins["cumulative"] = len_df_bins.proportion.cumsum()
len_df_bins["index"] = len_df_bins["index"].astype("str")

trace = go.Bar(x=len_df_bins["index"], y=len_df_bins["cumulative"])
fig = go.Figure(data=[trace])
fig.show(renderer = "colab")

As we can see from the cumulative density function, 80% of the dataset has its descriction under 200 words. So, we will use this value as the standard description lenght, and also add a minimum threshold to grant a least a few words to determinate the genre.

In [7]:
min_desc_length=6
max_desc_length=200

book_data_train=book_data_train[(book_data_train.clean_desc.str.split().apply(len)>min_desc_length)].reset_index(drop=True)
book_data_test = book_data_test[(book_data_test.clean_desc.str.split().apply(len)>min_desc_length)].reset_index(drop=True)

## Tokenization

The next step is to convert the words into integers that can be passed to the model

In [8]:
vocabulary = set()

def add_to_vocab(df, vocabulary):
  for i in df.clean_desc:
    for word in i.split():
      vocabulary.add(word)
  return vocabulary

vocabulary=add_to_vocab(book_data_train, vocabulary)

# mapping to token, starting with 1 cuz 0 will be for descriptions with less than 200 words
vocab_dict = {word: token+1 for token, word in enumerate(list(vocabulary))}
# mapping from token to word
token_dict = {token+1: word for token, word in enumerate(list(vocabulary))}

assert token_dict[1]==token_dict[vocab_dict[token_dict[1]]]

In [9]:
def tokenizer (desc, vocab_dict, max_desc_length):
  a = [vocab_dict[i] if i in vocab_dict else 0 for i in desc.split()]
  b = [0] * max_desc_length
  if len(a) < max_desc_length:
    return np.asarray(b[:max_desc_length - len(a)]+a).squeeze()
  else:
    return np.asarray(a[:max_desc_length]).squeeze()

book_data_train["desc_tokens"] = book_data_train["clean_desc"].apply(tokenizer, args=(vocab_dict, max_desc_length))
book_data_test["desc_tokens"] = book_data_test["clean_desc"].apply(tokenizer, args=(vocab_dict, max_desc_length))

### Stratifying training-validation

Training and validation datasets.

In [10]:
def stratified_split (df, target, val_percent=0.2):
  classes = list(df[target].unique())
  train_idxs, val_idxs = [], []
  for c in classes:
    idx = list(df[df[target]==c].index)
    np.random.shuffle(idx)
    val_size = int(len(idx)*val_percent)
    val_idxs += idx[:val_size]
    train_idxs += idx[val_size:]
  return train_idxs, val_idxs

_, sample_idxs = stratified_split(book_data_train, "binary_genre", 0.1)

train_idxs, val_idxs = stratified_split(book_data_train, "binary_genre", 0.2)
sample_train_idxs, sample_val_idxs = stratified_split(book_data_train[book_data_train.index.isin(sample_idxs)], "binary_genre", 0.2)

In [11]:
classes=list(book_data_train.binary_genre.unique())

sampling=False

x_train=np.stack(book_data_train[book_data_train.index.isin(sample_train_idxs if sampling else train_idxs)]['desc_tokens'])
y_train=book_data_train[book_data_train.index.isin(sample_train_idxs if sampling else train_idxs)]['binary_genre'].apply(lambda x:classes.index(x))

x_val=np.stack(book_data_train[book_data_train.index.isin(sample_val_idxs if sampling else val_idxs)]['desc_tokens'])
y_val=book_data_train[book_data_train.index.isin(sample_val_idxs if sampling else val_idxs)]['binary_genre'].apply(lambda x:classes.index(x))

x_test=np.stack(book_data_test['desc_tokens'])
y_test=book_data_test['binary_genre'].apply(lambda x:classes.index(x))

## Model

The next step is to build a recurrent neural network to process the tokenized descriptions and classify them as fiction or nonficion using Keras API.

In [12]:
model = Sequential()
model.add(Embedding(len(vocabulary)+1, output_dim=200, input_length=max_desc_length))
model.add(LSTM(200, return_sequences=True))
model.add(Dropout(0.5))
model.add(LSTM(200))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print(model.summary())

model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=10, batch_size=256)
score = model.evaluate(x_test, y_test, batch_size= 16)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning:

Argument `input_length` is deprecated. Just remove it.



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 117ms/step - accuracy: 0.8327 - loss: 0.3862 - val_accuracy: 0.9200 - val_loss: 0.2096
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 114ms/step - accuracy: 0.9544 - loss: 0.1317 - val_accuracy: 0.9307 - val_loss: 0.1919
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 115ms/step - accuracy: 0.9784 - loss: 0.0677 - val_accuracy: 0.9155 - val_loss: 0.2176
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 115ms/step - accuracy: 0.9867 - loss: 0.0459 - val_accuracy: 0.8952 - val_loss: 0.3523
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 116ms/step - accuracy: 0.9912 - loss: 0.0314 - val_accuracy: 0.9242 - val_loss: 0.2888
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 117ms/step - accuracy: 0.9971 - loss: 0.0116 - val_accuracy: 0.9239 - val_loss: 0.3503
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - accuracy: 0.9981 - loss: 0.0074 - val_accuracy: 0.9212 - val_loss: 0.3727
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - accuracy: 0.9979 - loss: 0.0077 - val